###Task 3.1: Features, Split & Baseline

In [16]:
import pandas as pd

df=pd.read_csv("/content/appointments_clean.csv")

df.drop(columns=['appointment_id','patient_id','clinic_id','booked_on','appt_date'],inplace=True)

In [17]:
from sklearn.model_selection import train_test_split

df['reminder_sent']=df['reminder_sent'].map({"Yes":1,"No":0})
df['gender']=df['gender'].map({"F":1,"M":0})
df['status']=df['status'].map({'No-Show':0,'Attended':1})
x=df.drop(columns="no_show",axis=1)

x=pd.get_dummies(x,columns=['booking_channel' ,'visit_type','booking_channel'],dtype=int)

y=df['no_show']

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)
x

,age,gender,distance_km,fee_inr,reminder_sent,prior_no_shows,status,lead_days,booking_channel_App,booking_channel_Phone,booking_channel_Walk-In,visit_type_Consultation,visit_type_Follow-up,booking_channel_App,booking_channel_Phone,booking_channel_Walk-In
0,60,1,18.0,850,1,1,0,1.0,0,1,0,1,0,0,1,0
1,39,0,12.5,350,0,2,1,10.0,0,0,1,0,1,0,0,1
2,63,1,4.1,850,0,0,1,6.0,0,0,1,1,0,0,0,1
3,52,1,7.5,750,1,0,1,4.0,1,0,0,1,0,1,0,0
4,64,1,19.0,700,0,0,0,40.0,0,1,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
951,26,0,24.8,900,0,0,1,1.0,1,0,0,1,0,1,0,0
952,20,0,13.0,700,0,1,1,22.0,0,1,0,1,0,0,1,0
953,77,0,17.4,400,1,3,0,4.0,1,0,0,0,1,1,0,0
954,58,0,3.2,850,0,1,1,11.0,0,0,1,1,0,0,0,1


In [38]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score,roc_auc_score
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

model=LogisticRegression(max_iter=1000)
model.fit(x_train,y_train)

y_pred=model.predict(x_test)

accuracy_log=accuracy_score(y_test,y_pred)
precision_log=precision_score(y_test,y_pred)
recall_log=recall_score(y_test,y_pred)
f1_log=f1_score(y_test,y_pred)
roc_auc_log=roc_auc_score(y_test,y_pred)

print("Accuracy:",accuracy_log)
print("Precision:",precision_log)
print("Recall:",recall_log)
print("F1 Score:",f1_log)
print("ROC AUC Score:",roc_auc_log)

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC AUC Score: 1.0


In [39]:
import numpy as np

print(f"Coefficients: {model.coef_}")
print("Odd ratios:\n")
print(np.exp(model.coef_))

feature_cols = list(x.columns)
idx = feature_cols.index("reminder_sent")

coef_rem = model.coef_[0][idx]
scale_rem = scaler.scale_[idx]

print("Scaled odds ratio (1 SD):", np.exp(coef_rem))
print("No -> Yes odds ratio:", np.exp(coef_rem / scale_rem))

Coefficients: [[-0.09488328 -0.02521718  0.11970441  0.13144679 -0.19081744  0.24607393
  -4.13853281  0.22853533 -0.00892001  0.05864012 -0.05395745  0.02149519
  -0.02149519 -0.00892001  0.05864012 -0.05395745]]
Odd ratios:

[[0.90947908 0.97509811 1.12716362 1.14047722 0.82628342 1.27899412
  0.01594623 1.25675793 0.99111965 1.06039356 0.94747242 1.02172787
  0.97873419 0.99111965 1.06039356 0.94747242]]
Scaled odds ratio (1 SD): 0.8262834184633254
No -> Yes odds ratio: 0.8262834184633254


###Task 3.3: Decision Tree & Overfitting Check

In [40]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score,roc_auc_score
model_tree=DecisionTreeClassifier(random_state=42)
model_tree.fit(x_train,y_train)
print(f"Training accuracy:{model_tree.score(x_train,y_train)}")
print(f"Testing accuracy:{model_tree.score(x_test,y_test)}")

Training accuracy:1.0
Testing accuracy:1.0


In [41]:
model=DecisionTreeClassifier(max_depth=3,random_state=42)
model_tree.fit(x_train,y_train)
print(f"Training accuracy:{model_tree.score(x_train,y_train)}")
print(f"Testing accuracy:{model_tree.score(x_test,y_test)}")
accuracy_tree=accuracy_score(y_test,y_pred)
precision_tree=precision_score(y_test,y_pred)
recall_tree=recall_score(y_test,y_pred)
f1_tree=f1_score(y_test,y_pred)
roc_auc_tree=roc_auc_score(y_test,y_pred)

print("Accuracy:",accuracy_tree)
print("Precision:",precision_tree)
print("Recall:",recall_tree)
print("F1 Score:",f1_tree)
print("ROC AUC Score:",roc_auc_tree)

Training accuracy:1.0
Testing accuracy:1.0
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC AUC Score: 1.0


In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score,roc_auc_score
model_forest=RandomForestClassifier(n_estimators=100, random_state=42)
model_forest.fit(x_train,y_train)
y_pred=model_forest.predict(x_test)
accuracy_forest=accuracy_score(y_test,y_pred)
precision_forest=precision_score(y_test,y_pred)
recall_forest=recall_score(y_test,y_pred)
f1_forest=f1_score(y_test,y_pred)
roc_auc_forest=roc_auc_score(y_test,y_pred)

print("Accuracy:",accuracy_forest)
print("Precision:",precision_forest)
print("Recall:",recall_forest)
print("F1 Score:",f1_forest)
print("ROC AUC Score:",roc_auc_forest)

results=[
    {"Model":"Logistic Regression","Accuracy":accuracy_log,"Precision":precision_log,"Recall":recall_log,"F1 Score":f1_log,"ROC AUC Score":roc_auc_log},
    {"Model":"Decision Tree","Accuracy":accuracy_tree,"Precision":precision_tree,"Recall":recall_tree,"F1 Score":f1_tree,"ROC AUC Score":roc_auc_tree},
    {"Model":"Random Forest","Accuracy":accuracy_forest,"Precision":precision_forest,"Recall":recall_forest,"F1 Score":f1_forest,"ROC AUC Score":roc_auc_forest}
]
for i in results:
  print(i)

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC AUC Score: 1.0
{'Model': 'Logistic Regression', 'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1 Score': 1.0, 'ROC AUC Score': np.float64(1.0)}
{'Model': 'Decision Tree', 'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1 Score': 1.0, 'ROC AUC Score': np.float64(1.0)}
{'Model': 'Random Forest', 'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1 Score': 1.0, 'ROC AUC Score': np.float64(1.0)}


In [49]:
from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)
from sklearn.pipeline import Pipeline


# --------------------------------
# 1. Hyperparameter tuning
# --------------------------------

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 8, None]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid_search.fit(x_train, y_train)

best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best Parameters:", best_params)
print("Best CV ROC-AUC:", grid_search.best_score_)


# --------------------------------
# 2. Compare models using CV
# --------------------------------

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

models_cv = {
    "LogReg": lr_pipe,
    "Depth-3 Tree": DecisionTreeClassifier(
        max_depth=3, random_state=42
    ),
    "RF": RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    "Tuned RF": best_model
}

print("\nModel Comparison (5-Fold CV):")

for name, est in models_cv.items():

    scores = cross_val_score(
        est,
        x_train,
        y_train,
        cv=5,
        scoring='roc_auc',
        n_jobs=-1
    )

    print(f"{name} CV ROC-AUC: {scores.mean():.4f} "
          f"+/- {scores.std():.4f}")


# --------------------------------
# 3. Out-of-fold probabilities
# --------------------------------

oof_probs = cross_val_predict(
    best_model,
    x_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("\nThreshold Comparison:")

for t in [0.5, 0.35]:

    oof_pred_t = (oof_probs >= t).astype(int)

    print(
        f"Threshold {t} -> "
        f"F1: {f1_score(y_train, oof_pred_t, zero_division=0):.4f}, "
        f"Flagged: {oof_pred_t.mean():.2%}"
    )


# --------------------------------
# 4. Final test evaluation
# --------------------------------

# Use the tuned model
probs_test = best_model.predict_proba(x_test)[:, 1]

# Selected classification threshold
threshold = 0.35

y_pred = (probs_test >= threshold).astype(int)

accuracy_forest = accuracy_score(y_test, y_pred)

precision_forest = precision_score(
    y_test, y_pred, zero_division=0
)

recall_forest = recall_score(
    y_test, y_pred, zero_division=0
)

f1_forest = f1_score(
    y_test, y_pred, zero_division=0
)

roc_auc_forest = roc_auc_score(y_test, probs_test)


# --------------------------------
# 5. Print evaluation metrics
# --------------------------------

print("\nFinal Tuned Random Forest Results:")
print("Accuracy:", accuracy_forest)
print("Precision:", precision_forest)
print("Recall:", recall_forest)
print("F1 Score:", f1_forest)
print("ROC AUC Score:", roc_auc_forest)


# --------------------------------
# 6. Save predictions to CSV
# --------------------------------

results = pd.DataFrame({
    "y_true": y_test,
    "risk_prob": probs_test,
    "no_show_pred": y_pred
})

results.to_csv("test_risk_scores.csv", index=False)

print("\nTest predictions saved to test_risk_scores.csv")


Best Parameters: {'max_depth': 3, 'n_estimators': 50}
Best CV ROC-AUC: 1.0

Model Comparison (5-Fold CV):
LogReg CV ROC-AUC: 1.0000 +/- 0.0000
Depth-3 Tree CV ROC-AUC: 1.0000 +/- 0.0000
RF CV ROC-AUC: 1.0000 +/- 0.0000
Tuned RF CV ROC-AUC: 1.0000 +/- 0.0000

Threshold Comparison:
Threshold 0.5 -> F1: 1.0000, Flagged: 21.20%
Threshold 0.35 -> F1: 1.0000, Flagged: 21.20%

Final Tuned Random Forest Results:
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC AUC Score: 1.0

Test predictions saved to test_risk_scores.csv
